Trabalho de Viualização Computacional 2026

Integrantes:


*   Daniela Costa da Silva 14613625
*   Isaac Ferreira Souza Santos 15637912
*   Thiago Oliveira dos Santos 13696220
*   Anna Lucia Moro Lanzuolo 13864436

<a id="1- Importação das bibliotecas"></a>
## 1. BIBLIOTECAS

Importando as bibliotecas necessárias para as análises

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import spearmanr
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
import plotly.express as px

<a id="2- Importação dos demais datasets"></a>
## 2. IMPORTAÇÃO DOS DATASETS DOS DEMAIS INSTITUTOS PARA A SEGUNDA PARTE DO TRABALHO

Importando datasets do repositório no GitHub.

Aqui vale explicar:


*   A coluna Tabela de Origem representa os departamentos no dataset.
*   As colunas Documentos Total na Consulta e Citações Totais representam a quantidade de artigos publicados no total por esse pesquisador naquele ano e as citações totais do mesmo, respectivamente.
*   O Índice H mostra o equilibrio entre produtividade e o impacto, ou seja, um pesquisador tem índice h = 30 se ele tiver 30 artigos citados pelo menos 30 vezes.
*  O Índice i10 representa quantos artigos aquele pesquisador pubicou que foram citados mais de 10 vezes.



In [2]:
url_icmc = "https://raw.githubusercontent.com/Dani-ela-git/Visualization-Data/refs/heads/main/dataset.csv"
df_icmc = pd.read_csv(url_icmc)
#df.head(5)

url_eesc = "https://raw.githubusercontent.com/Dani-ela-git/Visualization-Data/refs/heads/main/DatasetEESC.csv"
df_eesc = pd.read_csv(url_eesc)
#df_eesc.head(5)

url_iqsc = "https://raw.githubusercontent.com/Dani-ela-git/Visualization-Data/refs/heads/main/DatasetIQSC.csv"
df_iqsc = pd.read_csv(url_iqsc)
#df_iqsc.head(5)

url_ifsc = "https://raw.githubusercontent.com/Dani-ela-git/Visualization-Data/refs/heads/main/DatasetIFSC.csv"
df_ifsc = pd.read_csv(url_ifsc)
df_ifsc.head(5)

,ano,numero_usp,nome,identificador_google_scholar,documentos total na consulta,citacoes_totais,indice_h,departamento,data_consulta
0,2015,95341,Antonio Carlos Hernandes,l5HQq9cAAAAJ,1296.0,7477.0,42.0,FCM,2026-11-17
1,2015,8472872,Beni Hartmann,NaN,277.0,7671.0,47.0,FCM,2026-11-17
2,2015,79265,Carlos Antonio Ruggiero,NaN,62.0,251.0,6.0,FCM,2026-11-17
3,2015,486012,Cleber Renato Mendonça,NaN,1271.0,9472.0,48.0,FCM,2026-11-17
4,2015,265427,Cristina Kurachi,NaN,1076.0,8267.0,47.0,FCM,2026-11-17


<a id="4-Tratamento de dados"></a>
## 3. TRATAMENTO DE DADOS

Tratamento de dados para serem utilizados depois.Trata um dataset, e é reutilizado para cada dataset.



In [3]:
def processar_instituto(nome, dataframe, ano_referencia=None):
    """
    Trata um dataframe de um instituto e retorna:
    - df_ultimo: pesquisadores do ano de referência
    - resumo: agregado por departamento
    - ano_ref: ano usado
    """
    d = dataframe.copy()

    # Padroniza nomes de colunas removendo espaços e acentos comuns
    d.columns = d.columns.str.strip().str.lower()

    # Mapeamento robusto que funciona tanto se a coluna estiver em português quanto se já estiver tratada
    mapa = {
        "ano": "ano",
        "numero usp": "numero_usp",
        "numero_usp": "numero_usp",
        "nome do pesquisador": "nome",
        "nome": "nome",
        "identificador do google scholar": "identificador_google_scholar",
        "identificador_google_scholar": "identificador_google_scholar",
        "documentos total na consulta": "documentos_totais",
        "documentos_totais": "documentos_totais",
        "citações totais": "citacoes_totais",
        "citacoes_totais": "citacoes_totais",
        "índice h": "indice_h",
        "indice_h": "indice_h",
        "índice i10": "indice_i10",
        "indice_i10": "indice_i10",
        "tabela de origem": "departamento",
        "departamento": "departamento",
        "data da consulta": "data_consulta",
        "data_consulta": "data_consulta",
    }
    d = d.rename(columns=mapa)

    # Conversões
    num_cols = ["ano", "documentos_totais", "citacoes_totais", "indice_h"]
    if "indice_i10" in d.columns:
        num_cols.append("indice_i10")

    for c in num_cols:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    if "data_consulta" in d.columns:
        d["data_consulta"] = pd.to_datetime(d["data_consulta"], errors="coerce")

    d["nome"] = d["nome"].astype("string").str.strip()
    d["departamento"] = (
        d["departamento"].astype("string").str.strip().str.upper()
    )

    # Limpeza
    d = d.dropna(subset=["ano", "nome", "departamento"]).copy()

    # ID único do pesquisador
    id_gs = (
        d["identificador_google_scholar"]
        .astype("string").str.strip().replace("", pd.NA)
        if "identificador_google_scholar" in d.columns
        else pd.Series(pd.NA, index=d.index)
    )
    id_usp = (
        d["numero_usp"]
        .astype("string").str.strip().replace("", pd.NA)
        if "numero_usp" in d.columns
        else pd.Series(pd.NA, index=d.index)
    )
    d["_id_pesquisador"] = (
        ("GS:" + id_gs)
        .fillna("USP:" + id_usp)
        .fillna("NOME:" + d["nome"].str.casefold()
                + "|DEP:" + d["departamento"])
    )

    # Ano de referência
    ano_ref = int(d["ano"].max()) if ano_referencia is None else ano_referencia

    # Último registro por pesquisador no ano de referência
    df_ultimo = (
        d[d["ano"] == ano_ref]
        .sort_values("data_consulta")
        .drop_duplicates("_id_pesquisador", keep="last")
        .copy()
    )

    # Cadastrados por departamento
    cadastrados = (
        df_ultimo.groupby("departamento")["_id_pesquisador"]
        .nunique().rename("n_cadastrados")
    )

    # Válidos (com documentos e citações)
    df_validos = df_ultimo.dropna(
        subset=["documentos_totais", "citacoes_totais"]
    ).copy()

    # Resumo por departamento
    agg = {
        "n_pesquisadores": ("_id_pesquisador", "nunique"),
        "docs_total": ("documentos_totais", "sum"),
        "citacoes_total": ("citacoes_totais", "sum"),
        "h_medio": ("indice_h", "mean"),
        "h_max": ("indice_h", "max"),
    }
    # i10 só entra se existir
    if "indice_i10" in df_validos.columns and df_validos["indice_i10"].notna().any():
        agg["i10_total"] = ("indice_i10", "sum")

    resumo = df_validos.groupby("departamento").agg(**agg).reset_index()

    resumo = resumo.merge(cadastrados, on="departamento", how="left")
    resumo["sem_dados"] = resumo["n_cadastrados"] - resumo["n_pesquisadores"]

    resumo["citacoes_por_pesq"] = resumo["citacoes_total"] / resumo["n_pesquisadores"]
    resumo["docs_por_pesq"]    = resumo["docs_total"] / resumo["n_pesquisadores"]
    resumo["citacoes_por_doc"] = np.where(
        resumo["docs_total"] > 0,
        resumo["citacoes_total"] / resumo["docs_total"],
        np.nan,
    )

    resumo = resumo.sort_values("citacoes_total", ascending=False).reset_index(drop=True)
    resumo["instituto"] = nome

    print(f"[{nome}] ano_ref={ano_ref} | "
          f"{len(df_validos)} pesquisadores válidos | "
          f"i10 disponível: {'indice_i10' in df_validos.columns}")

    return df_ultimo, resumo, ano_ref

<a id="5-Gráficos"></a>
## 4. Função de gráficos universal

Função para reaproveitar para os gráficos de todos os datasets.



In [4]:
def plotar_resumo(resumo, nome, ano_ref):
    """Gera o painel 2x2 (ou 1x3 se não houver i10) + destaques."""

    # Indicadores base (sem i10)
    indicadores = [
        ("citacoes_total",    "Citações totais",              "#3274B9", "Citações"),
        ("citacoes_por_pesq", "Citações por pesquisador",     "#E58A32", "Média de citações"),
        ("h_medio",           "Índice h médio",               "#29966F", "Índice h"),
    ]
    # Adiciona i10 só se existir e tiver valores
    if "i10_total" in resumo.columns and resumo["i10_total"].notna().any():
        indicadores.append(
            ("i10_total", "Índice i10 total", "#8255AF", "Soma dos índices i10")
        )

    n = len(indicadores)
    ncols = 2
    nrows = (n + 1) // 2  # ceil

    fig = make_subplots(
        rows=nrows, cols=ncols,
        subplot_titles=[it[1] for it in indicadores],
        vertical_spacing=0.20 if nrows > 1 else 0.0,
        horizontal_spacing=0.14,
    )

    for i, (coluna, titulo, cor, eixo_y) in enumerate(indicadores):
        linha = i // ncols + 1
        col_sub = i % ncols + 1

        fig.add_trace(
            go.Bar(
                x=resumo["departamento"],
                y=resumo[coluna],
                marker_color=cor,
                customdata=resumo[["n_pesquisadores", "sem_dados"]].to_numpy(),
                hovertemplate=(
                    f"<b>Departamento: %{{x}}</b><br>{titulo}: %{{y:,.2f}}<br>"
                    "Pesquisadores analisados: %{customdata[0]}<br>"
                    "Pesquisadores sem dados: %{customdata[1]}<extra></extra>"
                ),
                showlegend=False,
            ),
            row=linha, col=col_sub,
        )
        fig.update_xaxes(title_text="Departamento", row=linha, col=col_sub)
        fig.update_yaxes(title_text=eixo_y, rangemode="tozero", row=linha, col=col_sub)

    fig.update_layout(
        title={
            "text": (
                f"Comparação bibliométrica — {nome}"
                f"<br><sup>Pesquisadores de {ano_ref} | Métricas coletadas em 2026</sup>"
            ),
            "x": 0.5, "xanchor": "center",
        },
        template="plotly_white",
        height=400 * nrows,
        bargap=0.28,
        font=dict(size=12),
        margin=dict(t=120, b=65, l=75, r=35),
        hoverlabel=dict(bgcolor="white", font_size=12),
    )
    fig.show()

    #Destaques
    maior_total = resumo.loc[resumo["citacoes_total"].idxmax()]
    maior_media = resumo.loc[resumo["citacoes_por_pesq"].idxmax()]
    maior_h     = resumo.loc[resumo["h_medio"].idxmax()]

    print(f"\nDESTAQUES — {nome}")
    print("Maior total de citações:            ", maior_total["departamento"])
    print("Maior média de citações/pesquisador:", maior_media["departamento"])
    print("Maior índice h médio:               ", maior_h["departamento"])
    print("-" * 60)



datasets = {
    "ICMC": df_icmc,
    "EESC": df_eesc,
    "IQSC": df_iqsc,
    "IFSC": df_ifsc,
}

resultados = {}

for nome, dset in datasets.items():
  df_ultimo, resumo, ano_ref = processar_instituto(nome, dset)
  display(resumo.round(2))
  plotar_resumo(resumo, nome, ano_ref)
  resultados[nome] = {"df_ultimo": df_ultimo, "resumo": resumo, "ano": ano_ref}

[ICMC] ano_ref=2025 | 120 pesquisadores válidos | i10 disponível: True


,departamento,n_pesquisadores,docs_total,citacoes_total,h_medio,h_max,i10_total,n_cadastrados,sem_dados,citacoes_por_pesq,docs_por_pesq,citacoes_por_doc,instituto
0,SME,35,5501.0,87582.0,19.26,78.0,1476.0,35,0,2502.34,157.17,15.92,ICMC
1,SCC,25,5037.0,58972.0,21.24,41.0,1251.0,25,0,2358.88,201.48,11.71,ICMC
2,SSC,22,4614.0,56156.0,23.18,36.0,1222.0,22,0,2552.55,209.73,12.17,ICMC
3,SMA,38,2342.0,20207.0,10.97,40.0,511.0,39,1,531.76,61.63,8.63,ICMC



DESTAQUES — ICMC
Maior total de citações:             SME
Maior média de citações/pesquisador: SSC
Maior índice h médio:                SSC
------------------------------------------------------------
[EESC] ano_ref=2025 | 174 pesquisadores válidos | i10 disponível: False


,departamento,n_pesquisadores,docs_total,citacoes_total,h_medio,h_max,n_cadastrados,sem_dados,citacoes_por_pesq,docs_por_pesq,citacoes_por_doc,instituto
0,SHS,22,5164.0,71249.0,27.89,68.0,24,2,3238.59,234.73,13.80,EESC
1,SEL,36,7257.0,62094.0,20.41,37.0,36,0,1724.83,201.58,8.56,EESC
2,SEP,18,3425.0,55327.0,23.47,41.0,19,1,3073.72,190.28,16.15,EESC
3,SEM,30,5161.0,45453.0,21.04,45.0,32,2,1515.10,172.03,8.81,EESC
4,SET,21,4049.0,35688.0,18.94,44.0,22,1,1699.43,192.81,8.81,EESC
5,SMM,14,2257.0,29399.0,23.46,41.0,15,1,2099.93,161.21,13.03,EESC
6,SAA,13,1767.0,13802.0,16.30,32.0,14,1,1061.69,135.92,7.81,EESC
7,STT,12,1939.0,11320.0,16.44,28.0,12,0,943.33,161.58,5.84,EESC
8,SGS,8,1178.0,9035.0,16.43,23.0,9,1,1129.38,147.25,7.67,EESC



DESTAQUES — EESC
Maior total de citações:             SHS
Maior média de citações/pesquisador: SHS
Maior índice h médio:                SHS
------------------------------------------------------------
[IQSC] ano_ref=2025 | 45 pesquisadores válidos | i10 disponível: False


,departamento,n_pesquisadores,docs_total,citacoes_total,h_medio,h_max,n_cadastrados,sem_dados,citacoes_por_pesq,docs_por_pesq,citacoes_por_doc,instituto
0,SQF,24,8825.0,141763.0,39.04,71.0,25,1,5906.79,367.71,16.06,IQSC
1,SQM,21,5193.0,65736.0,26.43,62.0,23,2,3130.29,247.29,12.66,IQSC



DESTAQUES — IQSC
Maior total de citações:             SQF
Maior média de citações/pesquisador: SQF
Maior índice h médio:                SQF
------------------------------------------------------------
[IFSC] ano_ref=2025 | 72 pesquisadores válidos | i10 disponível: False


,departamento,n_pesquisadores,docs_total,citacoes_total,h_medio,h_max,n_cadastrados,sem_dados,citacoes_por_pesq,docs_por_pesq,citacoes_por_doc,instituto
0,FCM,36,16901.0,198628.0,30.67,91.0,36,0,5517.44,469.47,11.75,IFSC
1,FCI,36,10555.0,148079.0,27.17,80.0,37,1,4113.31,293.19,14.03,IFSC



DESTAQUES — IFSC
Maior total de citações:             FCM
Maior média de citações/pesquisador: FCM
Maior índice h médio:                FCM
------------------------------------------------------------


<a id="4-Análise 1"></a>
## 4. Análise 1: qual departamento possui maior impacto científico?

Primeira análise feita no dataset é qual departamento do ICMC tem maior impacto científico.

A ideia nessa análise é: usar um boxplot para mostrar a mediana, quartis e outliers. Dessa forma, garantimos robustez mais do que se usarmos apenas a média, uma vez que não fica distorcido por valores extremos.

In [5]:

# Métricas base (sem i10)
metricas = [
    "citacoes_totais",
    "documentos_totais",
    "indice_h"
]

# Itera sobre os resultados de cada instituto
for nome, info in resultados.items():
    df_ultimo = info["df_ultimo"]

    # Deixa a ordem do eixo X padronizada pela mediana (maior para o menor)
    ordem_dep = (
        df_ultimo.groupby("departamento")["citacoes_totais"]
        .median()
        .sort_values(ascending=False)
        .index
    )

    for m in metricas:
        dados = df_ultimo.dropna(subset=["departamento", m]).copy()
        dados = dados[dados[m] >= 0].copy()

        # Citações totais tem uma escala muito grande, então aplicamos log(1+x)
        usar_log = (m == "citacoes_totais")

        if usar_log:
            dados["valor_grafico"] = np.log1p(dados[m])
        else:
            dados["valor_grafico"] = dados[m]

        fig = px.box(
            dados,
            x="departamento",
            y="valor_grafico",
            color="departamento",
            points="outliers",
            category_orders={"departamento": ordem_dep},
            color_discrete_sequence=px.colors.qualitative.Set2,
            hover_data={
                "nome": True,
                m: ":,.0f",
                "valor_grafico": False
            },
            labels={
                "departamento": "Departamento",
                "valor_grafico": m.replace("_", " ").capitalize(),
                "nome": "Pesquisador"
            },
            title=f"{nome} — Distribuição de {m.replace('_', ' ')} por departamento"
        )

        fig.update_layout(
            template="plotly_white",
            height=520,
            showlegend=False,
            xaxis_title="Departamento",
            yaxis_title=m.replace("_", " ").capitalize()
        )

        # Eixo Y com escala logarítmica "amigável" (10, 100, 1000...)
        if usar_log:
            maior_valor = dados[m].max()
            marcas = [0, 1]
            valor = 10
            while valor <= maior_valor:
                marcas.append(valor)
                valor *= 10

            fig.update_yaxes(
                tickmode="array",
                tickvals=np.log1p(marcas),
                ticktext=[f"{v:,}" for v in marcas],
                title_text=f"{m.replace('_', ' ').capitalize()} (escala log(1+x))"
            )

        fig.show()

        # Imprime a tabela de medianas
        print(f"\nMEDIANAS — {nome} — {m.replace('_', ' ').upper()}")
        medianas = (
            dados.groupby("departamento")[m]
            .median()
            .sort_values(ascending=False)
        )
        display(medianas.to_frame("Mediana").round(2))
        print("-" * 60)


MEDIANAS — ICMC — CITACOES TOTAIS


,Mediana
departamento,
SSC,2173.5
SCC,1027.0
SME,1022.0
SMA,237.0


------------------------------------------------------------



MEDIANAS — ICMC — DOCUMENTOS TOTAIS


,Mediana
departamento,
SSC,205.0
SCC,157.0
SME,114.0
SMA,44.5


------------------------------------------------------------


... (26 outputs left out)

------------------------------------------------------------



MEDIANAS — IFSC — INDICE H


,Mediana
departamento,
FCI,25.0
FCM,23.0


------------------------------------------------------------


<a id="5-Análise 2"></a>
## 5. Análise 2: produzir mais documentos significa receber mais citações?

Essa análise pode revelar que pesquisadores com poucos artigos publicados podem ter grandes impactos. Aqui, usamos um gráfico de disperção entre documentos e citações e também calculamos a correlação de Spearman.

In [8]:
# ============================================================
# 6. ANÁLISE 2: MATRIZ DE CORRELAÇÃO DE SPEARMAN
# ============================================================

from scipy.stats import spearmanr

# Métricas que vamos correlacionar (sem o i10)
metricas_corr = ["documentos_totais", "citacoes_totais", "indice_h"]

for nome, info in resultados.items():
    df_ultimo = info["df_ultimo"]

    # Filtra apenas os dados válidos para essas três métricas
    dados_corr = df_ultimo.dropna(subset=metricas_corr).copy()

    # Calcula a matriz de correlação de Spearman e os p-valores
    corr_matrix, p_values = spearmanr(dados_corr[metricas_corr])

    # Transforma em DataFrame para facilitar a visualização e nomes bonitos
    nomes_bonitos = {
        "documentos_totais": "Documentos Totais",
        "citacoes_totais": "Citações Totais",
        "indice_h": "Índice H"
    }

    df_corr = pd.DataFrame(
        corr_matrix,
        index=[nomes_bonitos[m] for m in metricas_corr],
        columns=[nomes_bonitos[m] for m in metricas_corr]
    )

    # Cria o Heatmap interativo
    fig = px.imshow(
        df_corr,
        text_auto=".2f", # Mostra os valores formatados com 2 casas decimais
        aspect="auto",
        color_continuous_scale="RdBu_r", # Vermelho (positivo) e Azul (negativo)
        zmin=-1, zmax=1, # Escala fixa de -1 a 1
        title=f"{nome} — Correlação de Spearman entre as métricas"
    )

    fig.update_layout(
        template="plotly_white",
        height=450,
        width=600,
        coloraxis_colorbar=dict(title="Correlação"),
        margin=dict(t=80, b=50, l=50, r=50)
    )

    # Ajusta o tamanho da fonte dos valores dentro dos quadrados
    fig.update_traces(textfont=dict(size=14))

    fig.show()

    # Imprime a matriz de forma tabular caso queiram analisar os números puros
    print(f"\nMATRIZ DE CORRELAÇÃO — {nome}")
    display(df_corr.round(3))
    print("-" * 60)

for nome, info in resultados.items():
    resumo = info["resumo"]

    # Encontra os índices máximos para cada métrica
    idx_maior_total = resumo["citacoes_total"].idxmax()
    idx_maior_media = resumo["citacoes_por_pesq"].idxmax()
    idx_maior_h     = resumo["h_medio"].idxmax()

    # Pega a linha inteira do departamento destaque
    maior_total = resumo.loc[idx_maior_total]
    maior_media = resumo.loc[idx_maior_media]
    maior_h     = resumo.loc[idx_maior_h]

    print(f"\n{'='*60}")
    print(f"DESTAQUES DA COMPARAÇÃO — {nome}")
    print(f"{'='*60}")
    print(f"Maior total de citações:             {maior_total['departamento']}")
    print(f"Maior média de citações por pesquisador: {maior_media['departamento']}")
    print(f"Maior índice h médio:                {maior_h['departamento']}")
    print(f"{'-'*60}")

for nome, info in resultados.items():
    resumo = info["resumo"]

    fig = px.bar(
        resumo,
        x="departamento",
        y="citacoes_total",
        color="departamento",
        text="citacoes_total",
        color_discrete_sequence=px.colors.qualitative.Set2,
        hover_data={
            "citacoes_total": ":,.0f",
            "n_pesquisadores": True,
            "departamento": False
        },
        labels={
            "departamento": "Departamento",
            "citacoes_total": "Total de citações",
            "n_pesquisadores": "Pesquisadores analisados"
        },
        title=f"{nome} — Citações totais por departamento"
    )

    fig.update_traces(
        texttemplate="%{y:,.0f}",
        textposition="outside"
    )

    fig.update_layout(
        template="plotly_white",
        height=500,
        showlegend=False,
        xaxis_title="Departamento",
        yaxis_title="Soma de citações",
        yaxis=dict(rangemode="tozero"),
        hovermode="closest"
    )

    fig.show()


MATRIZ DE CORRELAÇÃO — ICMC


,Documentos Totais,Citações Totais,Índice H
Documentos Totais,1.000,0.873,0.87
Citações Totais,0.873,1.000,0.97
Índice H,0.870,0.970,1.00


------------------------------------------------------------



MATRIZ DE CORRELAÇÃO — EESC


,Documentos Totais,Citações Totais,Índice H
Documentos Totais,1.000,0.757,0.768
Citações Totais,0.757,1.000,0.947
Índice H,0.768,0.947,1.000


------------------------------------------------------------


... (2 outputs left out)

------------------------------------------------------------



MATRIZ DE CORRELAÇÃO — IFSC


,Documentos Totais,Citações Totais,Índice H
Documentos Totais,1.000,0.774,0.783
Citações Totais,0.774,1.000,0.884
Índice H,0.783,0.884,1.000


------------------------------------------------------------

DESTAQUES DA COMPARAÇÃO — ICMC
Maior total de citações:             SME
Maior média de citações por pesquisador: SSC
Maior índice h médio:                SSC
------------------------------------------------------------

DESTAQUES DA COMPARAÇÃO — EESC
Maior total de citações:             SHS
Maior média de citações por pesquisador: SHS
Maior índice h médio:                SHS
------------------------------------------------------------

DESTAQUES DA COMPARAÇÃO — IQSC
Maior total de citações:             SQF
Maior média de citações por pesquisador: SQF
Maior índice h médio:                SQF
------------------------------------------------------------

DESTAQUES DA COMPARAÇÃO — IFSC
Maior total de citações:             FCM
Maior média de citações por pesquisador: FCM
Maior índice h médio:                FCM
------------------------------------------------------------


<a id="6-Análise 3"></a>
## 6. Análise 3: quais pesquisadores apresentam maior impacto por departamento?

In [9]:

for nome, info in resultados.items():
    df_ultimo = info["df_ultimo"]

    # Filtra pesquisadores que têm pelo menos documentos e citações válidos
    df_top = df_ultimo.dropna(subset=["documentos_totais", "citacoes_totais"]).copy()

    # Vamos fazer dois rankings: um por Citações Totais e outro por Índice H
    # Para não poluir a tela, faremos um gráfico com subplots (2 colunas)

    # Top 10 por Citações
    top_citacoes = df_top.nlargest(10, "citacoes_totais").sort_values("citacoes_totais", ascending=True)

    # Top 10 por Índice H (desempate por citações)
    top_h = df_top.sort_values(["indice_h", "citacoes_totais"], ascending=[True, True]).tail(10)

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Top 10 por Citações Totais", "Top 10 por Índice H"),
        horizontal_spacing=0.15
    )

    # Gráfico 1: Citações Totais
    fig.add_trace(
        go.Bar(
            y=top_citacoes["nome"],
            x=top_citacoes["citacoes_totais"],
            orientation="h",
            marker_color="#3274B9",
            text=top_citacoes["citacoes_totais"].apply(lambda x: f"{x:,.0f}"),
            textposition="outside",
            hovertemplate=(
                "<b>%{y}</b><br>"
                "Citações Totais: %{x:,.0f}<br>"
                "<extra></extra>"
            ),
            showlegend=False
        ),
        row=1, col=1
    )

    # Gráfico 2: Índice H
    fig.add_trace(
        go.Bar(
            y=top_h["nome"],
            x=top_h["indice_h"],
            orientation="h",
            marker_color="#29966F",
            text=top_h["indice_h"].apply(lambda x: f"{x:.0f}"),
            textposition="outside",
            hovertemplate=(
                "<b>%{y}</b><br>"
                "Índice H: %{x:.0f}<br>"
                "<extra></extra>"
            ),
            showlegend=False
        ),
        row=1, col=2
    )

    fig.update_layout(
        title={
            "text": f"{nome} — Destaques Individuais (Top 10)",
            "x": 0.5, "xanchor": "center"
        },
        template="plotly_white",
        height=600,
        margin=dict(t=100, b=50, l=50, r=50),
        bargap=0.3
    )

    # Ajusta os eixos X para não cortar o texto das barras
    fig.update_xaxes(showgrid=True, gridcolor="lightgrey")
    fig.update_yaxes(showticklabels=True)

    fig.show()

    # Imprime os dados em formato de tabela para os relatórios
    print(f"\n{'='*60}")
    print(f"TOP 5 PESQUISADORES — {nome}")
    print(f"{'='*60}")

    cols_mostrar = ["nome", "departamento", "documentos_totais", "citacoes_totais", "indice_h"]
    if "indice_i10" in df_top.columns and df_top["indice_i10"].notna().any():
        cols_mostrar.append("indice_i10")

    print("\nPor Citações Totais:")
    display(top_citacoes[cols_mostrar].tail(5).iloc[::-1].reset_index(drop=True))

    print("\nPor Índice H:")
    display(top_h[cols_mostrar].tail(5).iloc[::-1].reset_index(drop=True))

    print("-" * 60)


TOP 5 PESQUISADORES — ICMC

Por Citações Totais:


,nome,departamento,documentos_totais,citacoes_totais,indice_h,indice_i10
0,André Carlos Ponce de Leon Ferreira de Carvalho,SME,1150.0,24240.0,78.0,327.0
1,Francisco Aparecido Rodrigues,SME,228.0,12217.0,42.0,121.0
2,Francisco Louzada Neto,SME,388.0,8475.0,42.0,171.0
3,Agma Juci Machado Traina,SCC,683.0,8394.0,41.0,186.0
4,Caetano Traina Junior,SCC,343.0,6213.0,36.0,147.0



Por Índice H:


,nome,departamento,documentos_totais,citacoes_totais,indice_h,indice_i10
0,André Carlos Ponce de Leon Ferreira de Carvalho,SME,1150.0,24240.0,78.0,327.0
1,Francisco Aparecido Rodrigues,SME,228.0,12217.0,42.0,121.0
2,Francisco Louzada Neto,SME,388.0,8475.0,42.0,171.0
3,Agma Juci Machado Traina,SCC,683.0,8394.0,41.0,186.0
4,Alexandre Nolasco de Carvalho,SMA,298.0,5829.0,40.0,91.0


------------------------------------------------------------



TOP 5 PESQUISADORES — EESC

Por Citações Totais:


... (10 outputs left out)

,nome,departamento,documentos_totais,citacoes_totais,indice_h
0,Vanderlei Salvador Bagnato,FCM,1687.0,27821.0,84.0
1,Osvaldo Novais de Oliveira Junior,FCM,1106.0,26860.0,91.0
2,Luiz Vitor de Souza Filho,FCI,709.0,24821.0,80.0
3,Aion da Escossia Melo Viana,FCI,282.0,17122.0,59.0
4,Igor Polikarpov,FCM,962.0,16979.0,68.0



Por Índice H:


,nome,departamento,documentos_totais,citacoes_totais,indice_h
0,Osvaldo Novais de Oliveira Junior,FCM,1106.0,26860.0,91.0
1,Vanderlei Salvador Bagnato,FCM,1687.0,27821.0,84.0
2,Luiz Vitor de Souza Filho,FCI,709.0,24821.0,80.0
3,Igor Polikarpov,FCM,962.0,16979.0,68.0
4,Valtencir Zucolotto,FCM,1225.0,12228.0,64.0


------------------------------------------------------------


<a id="7-Análise 4"></a>
## 7. Análise 4: A produção científica se concentra em poucos pesquisadores?

Aqui, vamos calcular a porcentagem de citações que os 10% mais citados detêm e plotar a Curva de Lorenz. Se for muito curvada para baixo, então as citações é concntrada em poucos pesquisadores.

A curva de Lorenz plota a porcentagem acumulada da população (eixo X) contra a porcentagem acumulada de citações (eixo Y). Para isso, precisamos ordenar do menos citado para o mais citado.

Vamos mostrar também o Indice de Gini. O Gini resume a curva em um número: 0 é igualdade total, 1 é concentração total.

In [10]:

for nome, info in resultados.items():
    df_ultimo = info["df_ultimo"]

    # Filtra dados válidos
    df_disp = df_ultimo.dropna(subset=["documentos_totais", "citacoes_totais", "indice_h"]).copy()

    # Cria a coluna de tamanho para a bolinha (escala o Índice H para ficar visível)
    # Adicionamos um valor base para que mesmo quem tem H baixo apareça
    df_disp["tamanho_bolinha"] = df_disp["indice_h"] * 2 + 5

    # Gerando o gráfico de dispersão interativo
    fig = px.scatter(
        df_disp,
        x="documentos_totais",
        y="citacoes_totais",
        color="departamento",
        size="tamanho_bolinha",
        hover_name="nome",
        log_x=True, # Escala logarítmica no eixo X
        log_y=True, # Escala logarítmica no eixo Y
        color_discrete_sequence=px.colors.qualitative.Set2,
        hover_data={
            "documentos_totais": ":,.0f",
            "citacoes_totais": ":,.0f",
            "indice_h": True,
            "tamanho_bolinha": False
        },
        labels={
            "documentos_totais": "Documentos Totais (escala log)",
            "citacoes_totais": "Citações Totais (escala log)",
            "departamento": "Departamento",
            "indice_h": "Índice H"
        },
        title=f"{nome} — Relação entre Produtividade (Docs) e Impacto (Citações)"
    )

    fig.update_layout(
        template="plotly_white",
        height=600,
        legend_title_text="Departamento",
        margin=dict(t=80, b=50, l=50, r=50)
    )

    # Deixa as bolinhas com uma borda branca para destacar quando ficam sobrepostas
    fig.update_traces(marker=dict(line=dict(width=1, color='white')))

    fig.show()

    print(f"\nANÁLISE DE DISPERSÃO — {nome}")
    print("Pontos mais altos à direita = Alta produtividade e alto impacto.")
    print("Bolhas maiores = Índice H elevado.")
    print("-" * 60)


ANÁLISE DE DISPERSÃO — ICMC
Pontos mais altos à direita = Alta produtividade e alto impacto.
Bolhas maiores = Índice H elevado.
------------------------------------------------------------



ANÁLISE DE DISPERSÃO — EESC
Pontos mais altos à direita = Alta produtividade e alto impacto.
Bolhas maiores = Índice H elevado.
------------------------------------------------------------



ANÁLISE DE DISPERSÃO — IQSC
Pontos mais altos à direita = Alta produtividade e alto impacto.
Bolhas maiores = Índice H elevado.
------------------------------------------------------------



ANÁLISE DE DISPERSÃO — IFSC
Pontos mais altos à direita = Alta produtividade e alto impacto.
Bolhas maiores = Índice H elevado.
------------------------------------------------------------


<a id="8-Análise 5"></a>
## 8. Análise 5: Quantos pesquisadores entraram e quantos sairam nos ultimos 10 anos em cada departamento?

Aqui preciamos definir quem entrou e saiu ao decorrer dos anos. Quem entrou deve ter sua primeira data de publicação depois de 2016, pois não estava em 2015 e quem saiu deve ter sua ultima data de publicação até 2024, pois não estava em 2026.

Aqui talvez explique porque o SMA tem menos citações e desempenho inferior no impacto de produção científica.

In [11]:

for nome, info in resultados.items():
    df_ultimo = info["df_ultimo"]

    # Filtra dados válidos
    df_pareto = df_ultimo.dropna(subset=["citacoes_totais"]).copy()

    # Ordena do maior para o menor
    df_pareto = df_pareto.sort_values("citacoes_totais", ascending=False).reset_index(drop=True)

    # Calcula a porcentagem de cada pesquisador no total e o valor acumulado
    total_citacoes = df_pareto["citacoes_totais"].sum()

    if total_citacoes > 0:
        df_pareto["percentual_individual"] = (df_pareto["citacoes_totais"] / total_citacoes) * 100
        df_pareto["percentual_acumulado"] = df_pareto["percentual_individual"].cumsum()

        # Cria a posição no ranking para o eixo X (1, 2, 3...)
        df_pareto["rank"] = range(1, len(df_pareto) + 1)

        # Cria o gráfico de Pareto com dois eixos Y (barras e linha)
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        # 1. Barras: Citações Absolutas
        fig.add_trace(
            go.Bar(
                x=df_pareto["rank"],
                y=df_pareto["citacoes_totais"],
                name="Citações",
                marker_color="#3274B9",
                hovertext=df_pareto["nome"],
                hovertemplate="Pesquisador: %{hovertext}<br>Posição: %{x}<br>Citações: %{y:,.0f}<extra></extra>"
            ),
            secondary_y=False
        )

        # 2. Linha: Percentual Acumulado
        fig.add_trace(
            go.Scatter(
                x=df_pareto["rank"],
                y=df_pareto["percentual_acumulado"],
                name="% Acumulado",
                line=dict(color="#E58A32", width=3),
                mode="lines",
                hovertemplate="Posição: %{x}<br>Acumulado: %{y:.1f}%<extra></extra>"
            ),
            secondary_y=True
        )

        # 3. Linha vertical de referência (ex: Top 20%)
        top_20_rank = int(len(df_pareto) * 0.2) # Pega os primeiros 20% de pesquisadores
        if top_20_rank > 0:
            fig.add_vline(
                x=top_20_rank,
                line_dash="dash",
                line_color="gray",
                annotation_text="Top 20% dos Pesquisadores",
                annotation_position="top right"
            )

        fig.update_layout(
            title={
                "text": f"{nome} — Concentração de Impacto (Curva de Pareto)",
                "x": 0.5, "xanchor": "center"
            },
            template="plotly_white",
            height=500,
            hovermode="x unified", # Permite ver o acumulado daquela posição
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
        )

        # Ajusta os eixos
        fig.update_xaxes(title_text="Ranking de Pesquisadores (Mais citados para menos citados)")
        fig.update_yaxes(title_text="Citações Totais", secondary_y=False)
        fig.update_yaxes(title_text="Percentual Acumulado (%)", range=[0, 105], secondary_y=True)

        fig.show()

        # Imprime o insight textual
        if top_20_rank > 0:
            percentual_top_20 = df_pareto.loc[top_20_rank - 1, "percentual_acumulado"]
            print(f"\nPARETO — {nome}")
            print(f"Os top {top_20_rank} pesquisadores (20% do total) concentram {percentual_top_20:.1f}% de todas as citações do instituto.")
            print("-" * 60)


PARETO — ICMC
Os top 24 pesquisadores (20% do total) concentram 64.5% de todas as citações do instituto.
------------------------------------------------------------



PARETO — EESC
Os top 35 pesquisadores (20% do total) concentram 57.5% de todas as citações do instituto.
------------------------------------------------------------



PARETO — IQSC
Os top 9 pesquisadores (20% do total) concentram 49.8% de todas as citações do instituto.
------------------------------------------------------------



PARETO — IFSC
Os top 14 pesquisadores (20% do total) concentram 57.3% de todas as citações do instituto.
------------------------------------------------------------
